In [3]:
import polars as pl

results = pl.read_parquet("./reasoning-results.parquet")
from sklearn.metrics import cohen_kappa_score
from typing import Literal

In [4]:
results.columns
results["judge1_knowledge"].unique()

judge1_knowledge
i64
null
0
1
2
3
4


In [5]:
reasoning_scores = results.filter(pl.col("judge1_evaluated") & pl.col("judge2_evaluated"))

In [6]:
# Overall inter-rater agreement, unweighted
unweighted_agreement = {}
unweighted_agreement["interpretation"] = cohen_kappa_score(reasoning_scores["judge1_interpretation"], reasoning_scores["judge2_interpretation"])
unweighted_agreement["consistency"] = cohen_kappa_score(reasoning_scores["judge1_consistency"], reasoning_scores["judge2_consistency"])
unweighted_agreement["visual-evidence"] = cohen_kappa_score(reasoning_scores["judge1_visual_evidence"], reasoning_scores["judge2_visual_evidence"])

responses_with_knowledge = reasoning_scores.filter(pl.col("judge1_knowledge").is_not_null())
unweighted_agreement["knowledge"] = cohen_kappa_score(responses_with_knowledge["judge1_knowledge"], responses_with_knowledge["judge2_knowledge"])
print(unweighted_agreement)

{'interpretation': 0.222431943091438, 'consistency': 0.22960117141350367, 'visual-evidence': 0.10658843321614642, 'knowledge': 0.4409924616435499}


In [7]:
# Overall inter-rater agreement, quadratic weighting
quadratic_agreement = {}
quadratic_agreement["interpretation"] = cohen_kappa_score(reasoning_scores["judge1_interpretation"], reasoning_scores["judge2_interpretation"], weights="quadratic")
quadratic_agreement["consistency"] = cohen_kappa_score(reasoning_scores["judge1_consistency"], reasoning_scores["judge2_consistency"], weights="quadratic")
quadratic_agreement["visual-evidence"] = cohen_kappa_score(reasoning_scores["judge1_visual_evidence"], reasoning_scores["judge2_visual_evidence"], weights="quadratic")

responses_with_knowledge = reasoning_scores.filter(pl.col("judge1_knowledge").is_not_null())
quadratic_agreement["knowledge"] = cohen_kappa_score(responses_with_knowledge["judge1_knowledge"], responses_with_knowledge["judge2_knowledge"], weights="quadratic")
print(quadratic_agreement)

{'interpretation': 0.501043348851297, 'consistency': 0.3625996849513282, 'visual-evidence': 0.4075641071870202, 'knowledge': 0.8340197740137032}


In [8]:
# Overall inter-rater agreement, linear weighting
linear_agreement = {}
linear_agreement["interpretation"] = cohen_kappa_score(reasoning_scores["judge1_interpretation"], reasoning_scores["judge2_interpretation"], weights="linear")
linear_agreement["consistency"] = cohen_kappa_score(reasoning_scores["judge1_consistency"], reasoning_scores["judge2_consistency"], weights="linear")
linear_agreement["visual-evidence"] = cohen_kappa_score(reasoning_scores["judge1_visual_evidence"], reasoning_scores["judge2_visual_evidence"], weights="linear")

responses_with_knowledge = reasoning_scores.filter(pl.col("judge1_knowledge").is_not_null())
linear_agreement["knowledge"] = cohen_kappa_score(responses_with_knowledge["judge1_knowledge"], responses_with_knowledge["judge2_knowledge"], weights="linear")
print(linear_agreement)

{'interpretation': 0.36216874127454635, 'consistency': 0.3001754756817192, 'visual-evidence': 0.24575371498795728, 'knowledge': 0.6812353219258901}


In [9]:
# Exact agreement
exact_agreement = {}
exact_agreement["interpretation"] = (reasoning_scores["judge1_interpretation"] == reasoning_scores["judge2_interpretation"]).mean()
exact_agreement["consistency"] = (reasoning_scores["judge1_consistency"] == reasoning_scores["judge2_consistency"]).mean()
exact_agreement["visual-evidence"] = (reasoning_scores["judge1_visual_evidence"] == reasoning_scores["judge2_visual_evidence"]).mean()
responses_with_knowledge = reasoning_scores.filter(pl.col("judge1_knowledge").is_not_null())
exact_agreement["knowledge"] = (reasoning_scores["judge1_knowledge"] == reasoning_scores["judge2_knowledge"]).mean()
print(exact_agreement)

{'interpretation': 0.5471570397111913, 'consistency': 0.9600631768953068, 'visual-evidence': 0.335514440433213, 'knowledge': 0.5799438990182328}


In [12]:
# Let's investigate further, lets construct some confusion matrices
def construct_cm(df: pl.DataFrame, field: Literal["interpretation", "consistency", "knowledge", "visual_evidence"]):
    responses = None
    if field == "knowledge":
        responses = df.filter(pl.col("judge1_knowledge").is_not_null())
    else:
        responses = df
    judge1_field = f"judge1_{field}"
    judge2_field = f"judge2_{field}"
    return (
        df.group_by([judge1_field, judge2_field])
        .len()
        .pivot(
            values="len",
            index=judge1_field,
            on=judge2_field
        )
        .fill_null(0)
        .sort(judge1_field)
        .select([judge1_field, "0", "1", "2", "3", "4"])
        )


In [13]:
# Interpretation CM
interpretation_cm = construct_cm(reasoning_scores, field="interpretation")
print(interpretation_cm)

shape: (5, 6)
┌───────────────────────┬─────┬─────┬─────┬──────┬──────┐
│ judge1_interpretation ┆ 0   ┆ 1   ┆ 2   ┆ 3    ┆ 4    │
│ ---                   ┆ --- ┆ --- ┆ --- ┆ ---  ┆ ---  │
│ i64                   ┆ u32 ┆ u32 ┆ u32 ┆ u32  ┆ u32  │
╞═══════════════════════╪═════╪═════╪═════╪══════╪══════╡
│ 0                     ┆ 2   ┆ 4   ┆ 0   ┆ 0    ┆ 0    │
│ 1                     ┆ 4   ┆ 88  ┆ 14  ┆ 6    ┆ 0    │
│ 2                     ┆ 7   ┆ 113 ┆ 122 ┆ 46   ┆ 20   │
│ 3                     ┆ 2   ┆ 59  ┆ 168 ┆ 207  ┆ 44   │
│ 4                     ┆ 5   ┆ 67  ┆ 267 ┆ 1181 ┆ 2006 │
└───────────────────────┴─────┴─────┴─────┴──────┴──────┘


In [14]:
# Consistency CM
consistency_cm = construct_cm(reasoning_scores, field="consistency")
print(consistency_cm)

shape: (4, 6)
┌────────────────────┬─────┬─────┬─────┬─────┬──────┐
│ judge1_consistency ┆ 0   ┆ 1   ┆ 2   ┆ 3   ┆ 4    │
│ ---                ┆ --- ┆ --- ┆ --- ┆ --- ┆ ---  │
│ i64                ┆ u32 ┆ u32 ┆ u32 ┆ u32 ┆ u32  │
╞════════════════════╪═════╪═════╪═════╪═════╪══════╡
│ 1                  ┆ 0   ┆ 3   ┆ 1   ┆ 1   ┆ 3    │
│ 2                  ┆ 0   ┆ 5   ┆ 6   ┆ 5   ┆ 8    │
│ 3                  ┆ 0   ┆ 2   ┆ 6   ┆ 9   ┆ 10   │
│ 4                  ┆ 1   ┆ 12  ┆ 27  ┆ 96  ┆ 4237 │
└────────────────────┴─────┴─────┴─────┴─────┴──────┘


In [15]:
# Knowledge CM
knowledge_cm = construct_cm(reasoning_scores, field="knowledge")
print(knowledge_cm)

shape: (6, 6)
┌──────────────────┬─────┬─────┬─────┬─────┬─────┐
│ judge1_knowledge ┆ 0   ┆ 1   ┆ 2   ┆ 3   ┆ 4   │
│ ---              ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- │
│ i64              ┆ u32 ┆ u32 ┆ u32 ┆ u32 ┆ u32 │
╞══════════════════╪═════╪═════╪═════╪═════╪═════╡
│ 0                ┆ 226 ┆ 215 ┆ 51  ┆ 5   ┆ 1   │
│ 0                ┆ 0   ┆ 0   ┆ 0   ┆ 0   ┆ 0   │
│ 1                ┆ 12  ┆ 72  ┆ 50  ┆ 5   ┆ 0   │
│ 2                ┆ 1   ┆ 55  ┆ 239 ┆ 57  ┆ 1   │
│ 3                ┆ 0   ┆ 15  ┆ 116 ┆ 220 ┆ 22  │
│ 4                ┆ 2   ┆ 25  ┆ 71  ┆ 494 ┆ 897 │
└──────────────────┴─────┴─────┴─────┴─────┴─────┘


In [ ]:
# Visual-evidence CM
visual_evidence_cm = construct_cm(reasoning_scores, field="visual_evidence")
print(visual_evidence_cm)

In [ ]:
# What does it look like on a task by task basis
def compute_agreement(df: pl.DataFrame, weights: Literal["linear", "quadratic"] | None = None):
    agreements: dict[str, float] = {}
    agreements["interpretation"] = cohen_kappa_score(df["judge1_interpretation"], df["judge2_interpretation"], weights=weights)
    agreements["consistency"] = cohen_kappa_score(df["judge1_consistency"], df["judge2_consistency"], weights=weights)
    agreements["visual-evidence"] = cohen_kappa_score(df["judge1_visual_evidence"], df["judge2_visual_evidence"], weights=weights)

    knowledge = df.filter(pl.col("judge1_knowledge").is_not_null())
    if not knowledge.is_empty():
        agreements["knowledge"] = cohen_kappa_score(knowledge["judge1_knowledge"], knowledge["judge2_knowledge"], weights=weights)
    return agreements

def compute_reliability_for_task(df: pl.DataFrame, task=Literal["species_id", "fine_grained", "grounded_vqa", "localisation", "density"], weights: Literal["linear", "quadratic"] | None = None):
    # Might compute species ID differently later on
    responses_for_task = df.filter(pl.col("task") == task)
    if responses_for_task.is_empty():
        raise IndexError(f"Returned empty dataframe for {task}")
    return compute_agreement(responses_for_task, weights=weights)

def compute_reliability_on_task(df: pl.DataFrame, weights: Literal["linear", "quadratic"] | None = None):
    agreements: dict[str, dict[str, float]] = {}
    tasks = ["species_id", "fine_grained", "grounded_vqa", "localisation", "density"]
    for task in tasks:
        agreements[task] = compute_reliability_for_task(df, task, weights=weights) # pyright: ignore[reportArgumentType]
    return agreements

def task_pp(task: str, agreements: dict[str, float]):
    print(f"For task: {task}")
    for field, value in agreements.items():
        print(f"\t - {field}: {value:.4f}")

def all_tasks_pp(agreements: dict[str, dict[str, float]]):
    for task, agmts in agreements.items():
        task_pp(task, agmts)
per_task_agreements = compute_reliability_on_task(reasoning_scores, "quadratic")
all_tasks_pp(per_task_agreements)
    

Completed: species_id
Completed: fine_grained
Completed: grounded_vqa
Completed: localisation
Completed: density
For task: species_id
	 - interpretation: 0.4858
	 - consistency: 0.3412
	 - visual-evidence: 0.4201
	 - knowledge: 0.8440
For task: fine_grained
	 - interpretation: 0.3549
	 - consistency: 0.2308
	 - visual-evidence: 0.4756
For task: grounded_vqa
	 - interpretation: 0.6209
	 - consistency: 0.4478
	 - visual-evidence: 0.4818
	 - knowledge: 0.7302
For task: localisation
	 - interpretation: 0.5018
	 - consistency: 0.3352
	 - visual-evidence: 0.4953
	 - knowledge: 0.6681
For task: density
	 - interpretation: 0.5589
	 - consistency: 0.3848
	 - visual-evidence: 0.1906
